# Spring Pendulum Trainer

In this notebook you'll learn how to model a **spring pendulum** — a mass on a spring that swings under gravity. We'll build the physics from scratch, implement it numerically, and compare integration methods.

## Learning Objectives
- Understand Hooke's Law and simple harmonic motion
- Derive the equations of motion for a spring pendulum
- Implement Euler and RK4 integrators and compare accuracy
- Explore parameter space interactively

## 1. Hooke's Law

A spring exerts a restoring force proportional to its displacement from equilibrium:

$$F_{\text{spring}} = -k \, \Delta x$$

where:
- $k$ is the spring constant (N/m)
- $\Delta x = |\mathbf{r}| - L_0$ is the extension beyond the natural length $L_0$

For a mass on a spring with no other forces, Newton's second law gives:

$$m \ddot{x} = -k(x - L_0)$$

This is **simple harmonic motion** with angular frequency $\omega = \sqrt{k/m}$ and period $T = 2\pi/\omega$.

## 2. The Spring Pendulum Model

A spring pendulum combines a spring with a pendulum — the bob swings *and* bounces. In 3D, the forces acting on the bob at position $\mathbf{r}$ are:

**Spring restoring force:**
$$\mathbf{F}_{\text{spring}} = -k\,(|\mathbf{r}| - L_0)\,\hat{\mathbf{r}}$$

**Spring damping** (opposes radial velocity):
$$\mathbf{F}_{\text{damp}} = -b\,(\mathbf{v} \cdot \hat{\mathbf{r}})\,\hat{\mathbf{r}}$$

**Air resistance:**
$$\mathbf{F}_{\text{air}} = -c\,\mathbf{v}$$

**Gravity:**
$$\mathbf{F}_{\text{gravity}} = -mg\,\hat{\mathbf{y}}$$

The state vector is $\mathbf{s} = [x, y, z, v_x, v_y, v_z]$ and the derivatives are:

$$\frac{d\mathbf{s}}{dt} = [v_x, v_y, v_z, a_x, a_y, a_z]$$

where $\mathbf{a} = \mathbf{F}_{\text{net}} / m$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Physical parameters
k = 10.0          # Spring constant (N/m)
m = 1.0           # Mass (kg)
L0 = 2.0          # Natural length (m)
g = 9.8           # Gravity (m/s^2)
damping = 0.05    # Radial damping coefficient
air_res = 0.01    # Air resistance coefficient

print(f"Natural frequency: ω = {np.sqrt(k/m):.3f} rad/s")
print(f"Period of pure spring: T = {2*np.pi/np.sqrt(k/m):.3f} s")

## 3. Implementing the Derivatives

Let's implement the force calculation from first principles:

In [ ]:
def spring_pendulum_derivatives(state, t):
    """Compute derivatives for the spring pendulum."""
    pos = state[:3]
    vel = state[3:]
    
    # Distance from pivot
    distance = np.linalg.norm(pos)
    if distance < 1e-12:
        pos_hat = np.array([0.0, -1.0, 0.0])
        distance = 1e-12
    else:
        pos_hat = pos / distance
    
    # Spring extension
    extension = distance - L0
    
    # Forces
    f_spring = -k * extension * pos_hat
    f_damp = -damping * np.dot(vel, pos_hat) * pos_hat
    f_air = -air_res * vel
    f_gravity = np.array([0.0, -m * g, 0.0])
    
    # Acceleration
    acceleration = (f_spring + f_damp + f_air + f_gravity) / m
    
    return np.concatenate([vel, acceleration])

# Test with initial state
angle = 7 * np.pi / 8
initial_state = np.array([
    L0 * np.sin(angle),    # x
    -L0 * np.cos(angle),   # y
    0.0,                   # z
    0.0, 0.0, 0.5         # vx, vy, vz
])

deriv = spring_pendulum_derivatives(initial_state, 0.0)
print(f"Initial position: {initial_state[:3]}")
print(f"Initial acceleration: {deriv[3:]}")

## 4. Euler's Method vs RK4

**Euler's method** (first-order):
$$\mathbf{s}(t + \Delta t) = \mathbf{s}(t) + \Delta t \, f(\mathbf{s}, t)$$

**RK4** (fourth-order):
$$\mathbf{s}(t + \Delta t) = \mathbf{s}(t) + \frac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

where:
- $k_1 = f(\mathbf{s}, t)$
- $k_2 = f(\mathbf{s} + \frac{\Delta t}{2} k_1,\, t + \frac{\Delta t}{2})$
- $k_3 = f(\mathbf{s} + \frac{\Delta t}{2} k_2,\, t + \frac{\Delta t}{2})$
- $k_4 = f(\mathbf{s} + \Delta t\, k_3,\, t + \Delta t)$

In [ ]:
def euler_step(state, t, dt, derivs_fn):
    """Single Euler integration step."""
    return state + dt * derivs_fn(state, t)

def rk4_step(state, t, dt, derivs_fn):
    """Single RK4 integration step."""
    k1 = derivs_fn(state, t)
    k2 = derivs_fn(state + 0.5 * dt * k1, t + 0.5 * dt)
    k3 = derivs_fn(state + 0.5 * dt * k2, t + 0.5 * dt)
    k4 = derivs_fn(state + dt * k3, t + dt)
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

def simulate(state0, dt, n_steps, step_fn):
    """Run a simulation and collect trajectory."""
    states = np.zeros((n_steps + 1, len(state0)))
    states[0] = state0
    t = 0.0
    for i in range(n_steps):
        states[i + 1] = step_fn(states[i], t, dt, spring_pendulum_derivatives)
        t += dt
    return states

In [ ]:
# Compare Euler vs RK4
dt = 0.01
n_steps = 2000  # 20 seconds

trajectory_euler = simulate(initial_state, dt, n_steps, euler_step)
trajectory_rk4 = simulate(initial_state, dt, n_steps, rk4_step)

time = np.linspace(0, dt * n_steps, n_steps + 1)

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Position comparison
axes[0].plot(time, trajectory_euler[:, 0], label='Euler x', alpha=0.7)
axes[0].plot(time, trajectory_rk4[:, 0], label='RK4 x', alpha=0.7)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('x position (m)')
axes[0].set_title('Euler vs RK4: x-position')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Energy comparison (approximate — kinetic + gravitational potential + spring PE)
def compute_energy(states):
    energies = []
    for s in states:
        pos, vel = s[:3], s[3:]
        KE = 0.5 * m * np.dot(vel, vel)
        PE_grav = m * g * pos[1]  # y is up
        extension = np.linalg.norm(pos) - L0
        PE_spring = 0.5 * k * extension**2
        energies.append(KE + PE_grav + PE_spring)
    return np.array(energies)

E_euler = compute_energy(trajectory_euler)
E_rk4 = compute_energy(trajectory_rk4)

axes[1].plot(time, E_euler - E_euler[0], label='Euler ΔE', alpha=0.7)
axes[1].plot(time, E_rk4 - E_rk4[0], label='RK4 ΔE', alpha=0.7)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Energy drift (J)')
axes[1].set_title('Energy Conservation')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Euler energy drift after 20s: {E_euler[-1] - E_euler[0]:.4f} J")
print(f"RK4 energy drift after 20s: {E_rk4[-1] - E_rk4[0]:.6f} J")

## 5. Using the `physics_modeling` Package

The package provides a ready-made `SpringPendulum` class with configurable parameters and integrator:

In [ ]:
from physics_modeling.oscillators.spring_pendulum import (
    SpringPendulum,
    SpringPendulumConfig,
)

# Create simulation with RK4 integrator
config = SpringPendulumConfig(
    k=10.0,
    damping=0.05,
    mass=1.0,
    rest_length=2.0,
    g=9.8,
    air_resistance=0.01,
    initial_angle=7 * np.pi / 8,
    initial_velocity=(0.0, 0.0, 0.5),
    dt=0.01,
    integrator="rk4",
)

sim = SpringPendulum(config)
print(f"Initial state: {sim.state}")
print(f"State shape: {sim.state.shape}")

In [ ]:
# Run simulation and collect trajectory
n_steps = 3000
positions = np.zeros((n_steps, 3))

for i in range(n_steps):
    state = sim.step()
    positions[i] = state[:3]

# 3D trajectory plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot(positions[:, 0], positions[:, 2], positions[:, 1], lw=0.5, alpha=0.8)
ax.scatter([0], [0], [0], color='red', s=50, label='Pivot')
ax.set_xlabel('X (m)')
ax.set_ylabel('Z (m)')
ax.set_zlabel('Y (m)')
ax.set_title('Spring Pendulum 3D Trajectory')
ax.legend()
plt.show()

## 6. Parameter Exploration

Let's see how the spring constant $k$ affects the motion. Higher $k$ means a stiffer spring with faster oscillations:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

k_values = [2.0, 10.0, 50.0]

for ax, k_val in zip(axes, k_values):
    cfg = SpringPendulumConfig(
        k=k_val, damping=0.02, mass=1.0, rest_length=2.0,
        initial_angle=3 * np.pi / 4, initial_velocity=(0.0, 0.0, 0.0),
        dt=0.005, integrator="rk4",
    )
    s = SpringPendulum(cfg)
    
    pos = np.zeros((2000, 3))
    for i in range(2000):
        pos[i] = s.step()[:3]
    
    ax.plot(pos[:, 0], pos[:, 1], lw=0.5)
    ax.set_title(f'k = {k_val} N/m\nω = {np.sqrt(k_val):.1f} rad/s')
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Effect of Damping

Damping removes energy from the system. With enough damping, the bob spirals inward to its equilibrium position:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

damping_values = [0.0, 0.1, 0.5, 2.0]
colors = ['blue', 'green', 'orange', 'red']

for damp, color in zip(damping_values, colors):
    cfg = SpringPendulumConfig(
        k=10.0, damping=damp, mass=1.0, rest_length=2.0,
        air_resistance=0.0,
        initial_angle=3 * np.pi / 4, initial_velocity=(0.0, 0.0, 0.0),
        dt=0.01, integrator="rk4",
    )
    s = SpringPendulum(cfg)
    
    # Track distance from pivot over time
    distances = np.zeros(1500)
    for i in range(1500):
        state = s.step()
        distances[i] = np.linalg.norm(state[:3])
    
    t = np.arange(1500) * 0.01
    ax.plot(t, distances, color=color, label=f'damping = {damp}', alpha=0.8)

ax.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5, label='Rest length')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Distance from pivot (m)')
ax.set_title('Effect of Damping on Spring Oscillation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Exercise

**Task:** Modify the spring pendulum to include a *driving force* — a periodic external force applied to the bob.

Add a term:
$$\mathbf{F}_{\text{drive}} = F_0 \cos(\omega_d t)\, \hat{\mathbf{x}}$$

1. Implement a `driven_spring_derivatives(state, t)` function that includes this force
2. Simulate with $F_0 = 5.0$ and $\omega_d = \sqrt{k/m}$ (resonance)
3. Plot the amplitude over time — what happens at resonance with low damping?
4. Try $\omega_d$ slightly off resonance and observe *beats*

In [ ]:
# Your solution here
F0 = 5.0
omega_d = np.sqrt(k / m)  # Resonance frequency

def driven_spring_derivatives(state, t):
    """Spring pendulum with a periodic driving force."""
    # Start with the undriven derivatives
    deriv = spring_pendulum_derivatives(state, t)
    
    # Add driving force in the x-direction
    # F_drive = F0 * cos(omega_d * t) in x-direction
    # acceleration contribution = F_drive / m
    deriv[3] += F0 * np.cos(omega_d * t) / m  # ax component
    
    return deriv

# TODO: simulate and plot the results
# Hint: use simulate() with driven_spring_derivatives